In [ ]:
# ======================================================================
# EXTRAÇÃO DE ONTOLOGIA USANDO LLM - VERSÃO FUNCIONAL
# ======================================================================
import json
import re
from collections import defaultdict
from functools import reduce
from rdflib import Graph, Namespace, Literal
from rdflib.namespace import RDF, RDFS, OWL
import matplotlib.pyplot as plt
import networkx as nx
import requests

In [ ]:
# ======================================================================
# TIPOS E CONSTANTES
# ======================================================================

NAMESPACE = "http://example.org/ontology#"

API_URLS = {
    'deepseek': 'https://api.deepseek.com/v1/chat/completions',
    'claude': 'https://api.anthropic.com/v1/messages',
    'openai': 'https://api.openai.com/v1/chat/completions'
}

CORES_TIPO = {
    'Pessoa': '#FF6B6B',
    'Lugar': '#4ECDC4',
    'Organizacao': '#45B7D1',
    'Evento': '#95E1D3',
    'Data': '#FFA07A',
}

INDICADORES_CLASSE = {
    'Pessoa': ['pessoa', 'pessoas', 'indivíduo', 'cientista', 'engenheiro', 'clérigo'],
    'Lugar': ['cidade', 'país', 'lugar', 'local', 'condado', 'estado'],
    'Organizacao': ['empresa', 'organização', 'serviço', 'instituição', 'exército'],
    'Data': ['data', 'ano', 'dia', 'período']
}

PADROES_RELACAO = [
    (r'([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)\s+nasceu\s+em\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)', 'nasceu_em'),
    (r'pai\s+de\s+([A-Z][a-z]+)', 'pai_de'),
    (r'mãe\s+de\s+([A-Z][a-z]+)', 'mae_de'),
    (r'filho\s+de\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)', 'filho_de'),
    (r'filha\s+de\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)', 'filha_de'),
    (r'esposa\s+de\s+([A-Z][a-z]+)', 'esposa_de'),
]

In [ ]:
# ======================================================================
# FUNÇÕES IDENTIFICAÇÃO
# ======================================================================

In [ ]:
def identificar_classes(texto):
    texto_lower = texto.lower()
    
    classes_encontradas = [
        classe 
        for classe, palavras in INDICADORES_CLASSE.items()
        if any(palavra in texto_lower for palavra in palavras)
    ]
    
    return set(classes_encontradas)

In [ ]:
def identificar_pessoas(texto):
    padrao = r'(?:Rev\.|Sir)?\s*[A-Z][a-z]+(?:\s+[A-Z][a-z]+)+'
    pessoas = re.findall(padrao, texto)
    return set(pessoas)

In [ ]:
def identificar_lugares(texto):
    lugares_conhecidos = [
        'Londres', 'Maida Vale', 'Índia', 'Chatrapur', 'Bengala',
        'Grã-Bretanha', 'Países Baixos', 'Odisha', 'Índia britânica',
        'condado Tipperary', 'condado Longford', 'condado Clare'
    ]
    
    return set(lugar for lugar in lugares_conhecidos if lugar in texto)

In [ ]:
def identificar_organizacoes(texto):
    organizacoes_conhecidas = [
        'Serviço Civil Indiano', 'ICS', 'Ferrovias Madras',
        'Exército de Bengala', 'Hotel Colonnade'
    ]
    
    return set(org for org in organizacoes_conhecidas if org in texto)

In [ ]:
def identificar_datas(texto):
    padroes = [
        r'\(\d{4}[-–]\d{4}\)',
        r'\d{1,2} de \w+ de \d{4}'
    ]
    
    datas = []
    for padrao in padroes:
        datas.extend(re.findall(padrao, texto))
    
    return set(datas)

In [ ]:
def identificar_entidades(texto):
    return {
        'Pessoa': identificar_pessoas(texto),
        'Lugar': identificar_lugares(texto),
        'Organizacao': identificar_organizacoes(texto),
        'Data': identificar_datas(texto)
    }

In [ ]:
def identificar_relacoes(texto):
    relacoes = []
    
    for padrao, tipo_relacao in PADROES_RELACAO:
        matches = re.findall(padrao, texto, re.IGNORECASE)
        
        for match in matches:
            if isinstance(match, tuple):
                relacoes.append({
                    'sujeito': match[0],
                    'predicado': tipo_relacao,
                    'objeto': match[1]
                })
            else:
                relacoes.append({
                    'sujeito': 'Turing',
                    'predicado': tipo_relacao,
                    'objeto': match
                })
    
    return relacoes

In [ ]:
# ======================================================================
# FUNÇÕES PURAS - NORMALIZAÇÃO
# ======================================================================

In [ ]:
def normalizar_uri(texto):
    if not texto:
        return 'Entidade'
    
    texto = texto.strip()
    texto = re.sub(r'[^\w\s-]', '', texto)
    texto = texto.replace(' ', '_')
    texto = re.sub(r'^[\d_]+', '', texto)
    
    return texto if texto else 'Entidade'

In [ ]:
def criar_uri(namespace, nome):
    return namespace[normalizar_uri(nome)]


In [ ]:
# ======================================================================
# FUNÇÕES DE API
# ======================================================================

In [ ]:
def criar_prompt_sistema():
    return """
    Você é um especialista em extração de ontologias e modelagem de conhecimento.
    Sua tarefa é analisar textos e extrair:
    1. Classes (tipos de entidades)
    2. Instâncias (entidades concretas)
    3. Propriedades (relações entre entidades)
    
    Retorne SEMPRE em formato JSON válido.
    """

In [ ]:
def criar_prompt_usuario(texto):
    return f"""
    Analise o texto abaixo e extraia uma ontologia completa.
    
    TEXTO:
    {texto}
    
    INSTRUÇÕES:
    1. Identifique as CLASSES (tipos gerais como Pessoa, Lugar, Organização, etc.)
    2. Para cada classe, liste as INSTÂNCIAS (exemplos concretos)
    3. Identifique RELAÇÕES entre as entidades (ex: nasceu_em, trabalhou_em, pai_de)
    
    Retorne no formato JSON:
    {{
        "classes": ["Pessoa", "Lugar", ...],
        "entidades": {{
            "Pessoa": ["Alan Turing", "Julius Turing", ...],
            "Lugar": ["Londres", "Índia", ...]
        }},
        "relacoes": [
            {{"sujeito": "Alan Turing", "predicado": "nasceu_em", "objeto": "Londres"}},
            ...
        ]
    }}
    """

In [ ]:
def chamar_deepseek(api_key, prompt_usuario, prompt_sistema):
    headers = {
        'Content-Type': 'application/json',
        'Authorization': f'Bearer {api_key}'
    }
    
    payload = {
        'model': 'deepseek-chat',
        'messages': [
            {'role': 'system', 'content': prompt_sistema},
            {'role': 'user', 'content': prompt_usuario}
        ],
        'temperature': 0.3,
        'max_tokens': 4000
    }
    
    try:
        response = requests.post(
            API_URLS['deepseek'],
            headers=headers,
            json=payload,
            timeout=60
        )
        
        if response.status_code == 200:
            return response.json()['choices'][0]['message']['content']
        else:
            print(f"❌ Erro na API: {response.status_code}")
            return None
            
    except Exception as e:
        print(f"❌ Erro ao chamar DeepSeek: {e}")
        return None

In [ ]:
def processar_resposta_llm(resposta):
    if not resposta:
        return None
    
    try:
        # Tentar extrair JSON
        if '```json' in resposta:
            json_str = resposta.split('```json')[1].split('```')[0].strip()
        elif '```' in resposta:
            json_str = resposta.split('```')[1].split('```')[0].strip()
        else:
            json_str = resposta
        
        return json.loads(json_str)
        
    except json.JSONDecodeError as e:
        print(f"⚠️ Erro ao parsear JSON: {e}")
        return None

In [ ]:
# ======================================================================
# FUNÇÕES DE EXTRAÇÃO
# ======================================================================

In [ ]:
def extrair_ontologia_com_llm(texto, api_key):
    print("# ======================================================================")
    print("# CONSULTANDO LLM PARA ANÁLISE DO TEXTO")
    print("# ======================================================================")      
    prompt_sistema = criar_prompt_sistema()
    prompt_usuario = criar_prompt_usuario(texto)
    
    resposta = chamar_deepseek(api_key, prompt_usuario, prompt_sistema)
    dados = processar_resposta_llm(resposta)
    
    if dados:
        print("# Resposta recebida e processada")
        print("# ======================================================================")  
        return dados


In [ ]:
def converter_dados_para_estrutura(dados):
    classes = set(dados.get('classes', []))
    
    # Converter entidades para defaultdict de sets
    entidades = defaultdict(set)
    for classe, lista_ent in dados.get('entidades', {}).items():
        for ent in lista_ent:
            entidades[classe].add(ent)
    
    # Converter relações para tuplas
    relacoes = [
        (rel.get('sujeito', ''), rel.get('predicado', ''), rel.get('objeto', ''))
        for rel in dados.get('relacoes', [])
    ]
    
    return {
        'classes': classes,
        'entidades': entidades,
        'relacoes': relacoes
    }

In [ ]:
# ======================================================================
# FUNÇÕES RDF
# ======================================================================

In [ ]:
def criar_grafo_vazio():
    g = Graph()
    ns = Namespace(NAMESPACE)
    g.bind("", ns)
    g.bind("owl", OWL)
    return g

In [ ]:
def adicionar_classe(grafo, namespace, nome_classe):
    classe_uri = criar_uri(namespace, nome_classe)
    grafo.add((classe_uri, RDF.type, OWL.Class))
    grafo.add((classe_uri, RDFS.label, Literal(nome_classe, lang='pt')))
    return grafo

In [ ]:
def adicionar_classes(grafo, namespace, classes):
    return reduce(
        lambda g, classe: adicionar_classe(g, namespace, classe),
        classes,
        grafo
    )

In [ ]:
def adicionar_instancia(grafo, namespace, classe, nome_instancia):
    classe_uri = criar_uri(namespace, classe)
    instancia_uri = criar_uri(namespace, nome_instancia)
    
    grafo.add((instancia_uri, RDF.type, classe_uri))
    grafo.add((instancia_uri, RDFS.label, Literal(nome_instancia, lang='pt')))
    
    return grafo

In [ ]:
def adicionar_instancias(grafo, namespace, entidades):
    for classe, lista_ent in entidades.items():
        for entidade in lista_ent:
            grafo = adicionar_instancia(grafo, namespace, classe, entidade)
    
    return grafo

In [ ]:
def adicionar_propriedade(grafo, namespace, nome_propriedade):
    prop_uri = criar_uri(namespace, nome_propriedade)
    grafo.add((prop_uri, RDF.type, OWL.ObjectProperty))
    grafo.add((prop_uri, RDFS.label, Literal(nome_propriedade, lang='pt')))
    return grafo

In [ ]:
def adicionar_tripla(grafo, namespace, sujeito, predicado, objeto):
    if not sujeito or not predicado or not objeto:
        return grafo
    
    sujeito_uri = criar_uri(namespace, sujeito)
    predicado_uri = criar_uri(namespace, predicado)
    objeto_uri = criar_uri(namespace, objeto)
    
    grafo.add((sujeito_uri, predicado_uri, objeto_uri))
    
    return grafo

In [ ]:
def adicionar_relacoes(grafo, namespace, relacoes):
    propriedades = set()
    
    for sujeito, predicado, objeto in relacoes:
        if predicado and predicado not in propriedades:
            grafo = adicionar_propriedade(grafo, namespace, predicado)
            propriedades.add(predicado)
        
        grafo = adicionar_tripla(grafo, namespace, sujeito, predicado, objeto)
    
    return grafo, propriedades

In [ ]:
def criar_ontologia_rdf(classes, entidades, relacoes):
    print("# ======================================================================")
    print("# CRIANDO ONTOLOGIA RDF")
    print("# ======================================================================")    
    grafo = criar_grafo_vazio()
    namespace = Namespace(NAMESPACE)
    
    grafo = adicionar_classes(grafo, namespace, classes)
    for classe in entidades.keys():
        if classe not in classes:
            grafo = adicionar_classe(grafo, namespace, classe)
            classes.add(classe)
    
    grafo = adicionar_instancias(grafo, namespace, entidades)
    grafo, propriedades = adicionar_relacoes(grafo, namespace, relacoes)
    
    print(f"# Total de triplas ....: {len(grafo)}")
    print(f"# Classes .............: {len(classes)}")
    print(f"# Propriedades ........: {len(propriedades)}")
    print("# ======================================================================")    

    return grafo, namespace, propriedades

In [ ]:
# ======================================================================
# FUNÇÕES DE SAÍDA
# ======================================================================

In [ ]:
def salvar_ontologia(grafo, arquivo='ontologia_llm.ttl'):
    grafo.serialize(destination=arquivo, format='turtle')
    print(f"# Ontologia salva em: {arquivo}")

In [ ]:
def exibir_resumo(classes, entidades, relacoes):
    print("# ======================================================================")
    print("# RESUMO DA ONTOLOGIA EXTRAÍDA COM LLM")
    print("# ======================================================================")
    
    print("# CLASSES")
    print("# ======================================================================")    
    for classe in sorted(classes):
        qtd = len(entidades.get(classe, []))
        print(f"# {classe}: {qtd} instâncias")
    
    print("# ======================================================================") 
    print("# ENTIDADES POR CLASSE")
    print("# ======================================================================")    
    for classe in sorted(entidades.keys()):
        print(f"\n  {classe}:")
        lista_ent = sorted(list(entidades[classe])[:5])
        for ent in lista_ent:
            print(f"# {ent}")
        if len(entidades[classe]) > 5:
            print(f"    ... e mais {len(entidades[classe])-5}")

    print("# ======================================================================")     
    print("# RELAÇÕES")
    print("# ======================================================================")    
    for i, (s, p, o) in enumerate(relacoes[:10], 1):
        print(f"# {i}. {s} → {p} → {o}")
    if len(relacoes) > 10:
        print(f"# ... e mais {len(relacoes)-10} relações")
    print("# ======================================================================")         

In [ ]:
def exibir_resumo_classe (classes, entidades, relacoes):
    print("# ======================================================================")
    print("# RESUMO DA ONTOLOGIA EXTRAÍDA COM LLM - CLASSES")
    print("# ======================================================================")
 
    for classe in sorted(classes):
        qtd = len(entidades.get(classe, []))
        print(f"# {classe:.<30} : {qtd} instâncias")
    
    print("# ======================================================================")
 

In [ ]:
def exibir_resumo_entidade_classe (classes, entidades, relacoes):
    print("# ======================================================================")
    print("# RESUMO DA ONTOLOGIA EXTRAÍDA COM LLM - ENTIDADES POR CLASSE")
    print("# ======================================================================")
 
    for classe in sorted(entidades.keys()):
        lista_ent = sorted(list(entidades[classe])[:5])
        for ent in lista_ent:
            print(f"# {ent}")
        if len(entidades[classe]) > 5:
            print(f"#   ... e mais {len(entidades[classe])-5}")
    
    print("# ======================================================================")


In [ ]:
def exibir_resumo_relacoes (classes, entidades, relacoes):
    print("# ======================================================================")
    print("# RESUMO DA ONTOLOGIA EXTRAÍDA COM LLM - RELAÇÕES")
    print("# ======================================================================")

    for i, (s, p, o) in enumerate(relacoes[:10], 1):
        print(f"# {i}. {s} → {p} → {o}")
    if len(relacoes) > 10:
        print(f"# ... e mais {len(relacoes)-10} relações")
    print("# ======================================================================") 

In [ ]:
# ======================================================================
# FUNÇÃO PRINCIPAL (PIPELINE)
# ======================================================================

In [ ]:
texto_exemplo = """
Como mencionado, os poloneses foram os responsáveis por construir as primeiras bombas eletromecânicas. Elas obtiveram algum êxito em decifrar as mensagens criptografadas pela Enigma, no entanto, esse sucesso diminuiu drásticamente conforme a máquina alemã era aprimorada pelos engenheiros nazistas. Uma dessas implementações tornou-a indecifrável. No entanto, a captura de algumas chaves de criptografia possibilitou a quebra dos códigos nazistas. Foi nesse cenário que a programação feita por Alan Turing e sua equipe mostrou a sua utilidade.
O cientista da computação teve a ideia de implementar uma nova bomba eletromecânica, mais eficiente que as anteriores. Para tanto, programou-a prevendo a existência de trechos que se repetiam em grande parte das mensagens, além disso, soube como explorar uma grande falha na criptografia usada pela Enigma, da qual falaremos mais adiante.
A Enigma podia ser configurada em 26 modos de encriptação distintos, baseados na posição em que três eixos mecânicos eram conectados entre si. A letra presente no primeiro eixo determinava quais seriam as letras seguintes. Desse modo, se a primeira letra fosse A, por exemplo, quando se digitava a letra J, essa era substituída pela letra B, no entanto, caso a letra inicial fosse B, a letra J seria substituída pela letra C. A letra inicial de cada mensagem era, portanto, a chave criptográfica de cada mensagem, e, por isso, era trocada todos os dias, exatamente à meia-noite.
Alan Turing e sua equipe sabiam do modo como a posição dos eixos da Enigma mudava a encriptação das mensagens e também que essa configuração era alterada todos os dias. Em uma de suas tentativas de quebrar o código criptografado, Turing desenvolveu uma técnica pela qual a Bombe buscava “atacar” uma frase específica que se repetia na mesma posição, em todas as mensagens trocadas entre as forças armadas nazistas. Além disso, o cientista inglês também se beneficiou de uma grande falha da máquina alemã: ela não era capaz de criptografar uma mensagem sem alterar todas as suas letras.
Isso indicava que se a frase original contivesse a letra A, por exemplo, a letra criptografada jamais poderia ser A, independentemente de qual fosse a configuração em que a máquina estava. Baseada nisso, a Bombe procurava a frase que se repetia e combinava-a às outras 26 possibilidades de configuração, captadas ao longo do tempo, até que, em alguma dessas combinações, a mesma letra não se repetisse. A frase em questão, presente no final de todas as mensagens, era “Heil Hitler” (salve Hitler).
O maior legado deixado pelo matemático Alan Turing é, sem dúvidas, a invenção da máquina de Turing. Esta é um modelo teórico que pode ser usado para implementar todos os aspectos lógicos e matemáticos de um computador, independentemente de como ele venha a ser construído (mecânica ou eletronicamente, por exemplo).
A máquina de Turing foi criada em 1936, muito tempo antes da invenção dos computadores modernos. A maior parte dos nossos dispositivos eletrônicos, como celulares e computadores, são máquinas programáveis, que operam de acordo com os fundamentos da máquina de Turing. As calculadoras, por exemplo, operam como as primeiras máquinas de Turing, programadas para realizar cálculos simples.
Como citado, a participação de Turing na decifração da Enigma e na construção da bomba eletromecânica contribuiu para acelerar o final da Segunda Guerra Mundial, sendo responsável por salvar milhões de vidas. Além disso, nesse período, as tecnologias de criptografia e computação sofreram grandes avanços.
Além de ter construído as bases da computação moderna, Turing também desenvolveu os primeiros testes capazes de distinguir a inteligência artificial da inteligência humana. Atualmente os testes de Turing são usados em diversos sites e dispositivos, promovendo maior segurança para os seus usuários.
Como mencionado, a máquina criada por Turing, ou a sua bomba eletromecânica, era usada para decifrar mensagens emitidas pelas forças armadas alemãs e criptografadas por uma outra máquina chamada Enigma. Essas mensagens eram emitidas em forma de ondas de rádio, por isso eram facilmente interceptadas em Bletchley Park.
A Bombe era um enorme computador eletromecânico que pesava quase uma tonelada e tinha cerca de 1,80 m altura. Na parte frontal da máquina, havia 108 eixos, que eram agrupados em nove linhas com 12 espaços cilíndricos. Nesses cilindros eram encaixados os tambores, que, depois de programados manualmente, por meio de cartões com pequenos furos, giravam simultaneamente, combinando as letras de cada tambor com as mensagens captadas em agrupamentos de três letras.
Os tambores iniciavam a combinação em uma determinada letra e, ao final de cada ciclo, sua posição era incrementada para a próxima letra, assim, o ciclo era iniciado novamente. Ao todo, cada ciclo passava por um total de 17.576 posições diferentes. Agora, vamos entender como funcionava a máquina que codificava as comunicações realizadas pelos nazistas e que foi decifrada por meio do trabalho de Turing.
Alan Mathison Turing nasceu no dia 23 de junho de 1912, em um bairro residencial de Londres, capital da Inglaterra. Seu pai, Julius Mathison Turing, era um oficial que trabalhava na Madras Presidency, uma região administrativa criada pelos britânicos na Índia britânica (na época, a Índia era um território dependente da Inglaterra). Sua mãe, Ethel Sara Stoney, era filha de um engenheiro-chefe que também trabalhava nessa região.
Durante sua infância, Turing estudou em diversas escolas, tais como Hazelhurst Preparatory School e Sherborne School. Na Sherborne ingressou quando tinha 13 anos, e um episódio peculiar marcou sua entrada nela. No seu primeiro dia de aula, aconteceu uma greve geral na Grã-Bretanha que o impediu ir de trem. Turing resolveu, em sua bicicleta, percorrer os 100 km que separavam a escola de sua casa.
Alguns estudos feitos sobre a vida de Turing mostram que, em Sherborne, ele logo demonstrou grande interesse pela matemática, e, apesar de ser reconhecido atualmente como um gênio, algumas de suas notas eram apenas regulares. Em Sherborne, conheceu Christopher Morcom, o qual muitos acreditam ter sido seu primeiro amor.
Morcom, porém, faleceu em 1930, em decorrência de tuberculose bovina. A essa altura, Turing tinha 18 anos de idade. Anos depois, ele ingressou no curso de Matemática pela Universidade de Cambridge, e graduou-se em 1934, com honras. Depois disso, dedicou-se integralmente à matemática e à criptografia.
Em 1936, Turing apresentou uma teoria a respeito da construção de uma máquina capaz de realizar cálculos. Entre 1936 e 1938, estudou matemática e criptografia em Princeton, local em que obteve seu PhD. A partir de 1938, retornou à Inglaterra e passou a integrar uma organização do governo britânico, responsável por quebrar códigos e enigmas, chamada Government Code and Cypher School.
Com o começo da Segunda Guerra Mundial, em setembro de 1939, Turing ingressou no Bletchley Park, a instalação que reuniu grandes matemáticos e criptógrafos e que teve papel crucial na interceptação de mensagens enviadas pelos exércitos do Eixo (formado por Itália, Alemanha e Japão). Nessa instalação, houve uma intensa cooperação de cientistas ingleses, franceses e poloneses para decifrar o código utilizado pelos alemães e seus aliados.
Os alemães utilizavam a Enigma, uma máquina alemã que criptografava as mensagens que eram enviadas pelo exército e tornava-as quase indecifráveis. Alan Turing e outros matemáticos ingleses atuaram diretamente na quebra do código alemão, e, para isso, contaram com estudos realizados por três matemáticos poloneses, que atuaram entre 1932 e 1939.
Os franceses concederam aos britânicos chaves criptográficas utilizadas pelo Wehrmacht (exército alemão), e os poloneses deram-lhes máquinas Enigma. Em 1940, os britânicos conseguiram decodificar as primeiras mensagens enviadas pelos alemães, mas somente com a Bombe foi possível decodificá-las na velocidade demandada pela guerra.
A máquina utilizada pelos britânicos na decodificação das mensagens foi resultado do trabalho de Turing. Sua teoria inspirou-se nos estudos do polonês Marian Rejewski, e a execução do trabalho foi realizada pelo engenheiro Harold Keen. A importância de Turing dá-se porque foi ele que, ainda em 1939, afirmou que era possível construir um novo artefato capaz de quebrar o código alemão. Isso porque os poloneses já tinham desenvolvido uma bomba eletromecânica, mas como o princípio de criptoanálise da bomba polonesa era frágil, essa máquina logo se tornou obsoleta.
Alan Turing também foi o responsável por recrutar Tommy Flowers para o departamento de criptografia de Bletchley Park. Flowers, anos depois, acabou sendo o responsável pela construção do Colossus, uma importante máquina que decifrou o código de uma máquina de criptografia alemã chamada Lorenz.
A primeira bomba eletromecânica desenvolvida por meio dos estudos de Alan Turing ficou pronta em março de 1940 e foi nomeada Victory. As máquinas construídas com base nesse modelo desenvolvido por Alan Turing foram essenciais para os Aliados na Segunda Guerra Mundial, porque lhes deram uma vantagem estratégica extremamente importante: informação.
As mensagens decifradas pelas Bombe de Turing faziam parte do Ultra — departamento de inteligência britânico responsável por interceptar e decifrar as mensagens enviadas pelos sistemas de comunicação do Eixo. O Ultra teve atuação destacada na receptação de mensagens alemãs e contribuiu também para decifrar códigos japoneses.
Esse foi um departamento gigantesco, chegando a contar com seis mil trabalhadores, dos quais Alan Turing foi um dos nomes mais proeminentes. Por conta de sua importância, esse departamento era conhecido apenas pelo mais alto escalão do governo e do exército britânico. O líder soviético Josef Stalin, por exemplo, nunca soube como os britânicos conseguiam obter informações dos alemães.
O historiador Max Hastings argumentou que o Ultra (do qual as bombas eletromecânicas faziam parte) foi responsável por poupar a destruição de cerca de dois milhões de toneladas de embarcações britânicas, só no segundo semestre de 1941. Por meio das informações decodificadas pela máquina projetada por Turing, os britânicos conseguiram mapear a posição de embarcações alemãs, e, com isso, era possível desviar a rota das embarcações inglesas com antecipação.
Isso foi extremamente importante, porque, entre 1940 e 1941, o Reino Unido esteve lutando sozinho na guerra, uma vez que os franceses tinham sido rapidamente derrotados. A ilha britânica, isolada, acabou sendo cercada por submarinos alemães chamados de U-boats, que afundavam embarcações que levavam suprimento para o Reino Unido. Com a Bombe, foi possível contornar esse problema.
Em 1943, as máquinas desenvolvidas por Turing demonstravam a sua importância para a inteligência britânica, pois eram responsáveis por decifrar, a cada mês, cerca de 84 mil mensagens enviadas pelos alemães via Enigma.
O sistema de decodificação desenvolvido por Turing e sua equipe e o trabalho desenvolvido por meio do Ultra também contribuíram em batalhas realizadas no norte da África, na Grécia, na Normandia etc. Além de ajudar na vitória, Turing e sua invenção contribuiram para encurtar o tempo da Segunda Guerra e foram responsáveis por salvar a vida de milhões de pessoas.
Todo esse trabalho chegou ao seu ápice em 1941, quando por fim os matemáticos da Station X conseguiram criar a máquina mais poderosa no que dizia respeito a capacidade de decifrar códigos: a bomba eletromecânica, batizada como “The Bombe”. 
Ela funcionava a partir três rotores, com dois conjuntos de 26 contatos cada um. A máquina tinha 108 eixos agrupados em nove linhas, com 12 espaços cilíndricos.
Neles, estavam encaixados tambores programados por meio de cartões perfurados que giravam simultaneamente, permitindo que as letras recebidas fossem combinadas e testadas com outras letras já configuradas. A cada ciclo, a The Bombe passava por cerca de mais de 17 mil posições diferentes. 
Depois de interceptar a mensagem, a máquina era capaz de identificar que tipo de código havia sido utilizado. A partir disso, os cientistas configuravam uma réplica da Enigma com o mesmo tipo de cifragem, e assim obtinham a mensagem definitiva.  
As contribuições de Turing foram inestimáveis, especialmente para as estratégias navais, e como resultado fizeram com que o Reino Unido pudesse vencer grande parte das batalhas. 
Com o início da Guerra em 1939, Turing, juntamente com outros grandes matemáticos, foram para Bletchley Park, também conhecido como Station X, onde havia uma grande instalação militar focada na quebra de códigos e criptografia.
Essa base foi responsável por inúmeras interceptações de mensagens militares vindas do Japão, da Alemanha e da Itália, países que compunham o eixo durante a Guerra. 
Naquele momento, os alemães usavam uma máquina conhecida como Enigma, capaz de criptografar mensagens enviadas via telégrafo de forma quase indecifrável para os seus oponentes. A máquina continha 26 chaves capazes de embaralhar as palavras, conectando pequenos cabos que trocavam uma letra pela outra nas mensagens.
Ao apertar uma tecla, uma luz se acendia sobre outra tecla aleatória. O interessante é que a máquina não trocava uma letra pela mesma equivalente na mensagem, mesmo se ela se repetisse.
Por exemplo, ao escrever “palavra” na Enigma, os 3 ‘”As” poderiam tornar-se letras completamente diferentes. Estima-se que a máquina era capaz de gerar 159 quintilhões de formas diferentes de modificar uma mensagem. 
Mas essas cifragens também eram complicadas para os alemães. Todos os dias os códigos eram trocados, e existiam livros para guiar a decodificação das mensagens em uma outra unidade da máquina. Para acessar a mensagem, era preciso configurar a Enigma na mesma chave pela qual ela foi enviada. 
Os estudos para vencer a Enigma partiram da contribuição de matemáticos e criptoanalistas poloneses que já haviam conseguido decifrar algumas mensagens alemãs. A continuidade dos estudos deu frutos, e em 1940 os aliados conseguiram decodificar mensagens interceptadas do eixo, mas ainda com uma demora muito grande. 
Considerado um dos “pais da computação”, a vida de Alan Turing (1912-1954) foi muito além dos números. Como um grande matemático, Turing foi peça-chave para a vitória dos aliados na Segunda Guerra Mundial, revolucionou a tecnologia moderna e criou o primeiro teste que possibilitou a evolução das inteligências artificiais. 
Porém, outra parte de sua história foi bastante trágica, marcada pelo preconceito e perseguição por parte do governo britânico. Devido a Turing ser um homossexual assumido, o governo da Inglaterra o considerou criminoso e o obrigou a passar por tratamentos hormonais extremamente invasivos. Ele morreu aos 41 anos de idade, e teve sua história retratada em filmes, livros e documentários. 
Logo nos seus primeiros anos profissionais, Turing desenvolveu a Máquina de Turing, que desde então é utilizada em sistemas de calculadoras e outras máquinas mais simples, até os dias de hoje. Era um protótipo do computador, em que o sistema contava com um programa que determinava as tarefas a serem executadas.  
Além disso, ele também desenvolveu o Teste de Turing, usado para comparar respostas geradas por humanos e por computadores, dessa forma, era possível avaliar a capacidade de inteligência das máquinas. O teste foi muito importante para o desenvolvimento da inteligência artificial como vemos hoje.  
Uma de suas últimas contribuições foi na área da biologia matemática, em que ele descreveu a maneira pela qual os padrões naturais, como listras, manchas e espirais em animais, podem surgir naturalmente de um estado homogêneo e uniforme.
Alan Mathison Turing nasceu em 1912, em Paddington, uma área do condado de Londres, na Inglaterra. Desde pequeno ele já demonstrava interesse e curiosidade pela Ciência, especialmente quando ela envolvia números e lógica. 
Durante a sua infância, Turing frequentou duas instituições de ensino, a Hazelhurst Preparatory School e a Sherborne School. Apesar de hoje ele receber reconhecimento como se fosse um gênio, suas notas não eram extraordinárias, e ele demonstrava pouco interesse por matérias que não envolviam a matemática. 
Segundo relatos, foi na Sherborne School em que ele conheceu o seu primeiro amor e se entendeu como homossexual, aos 16 anos de idade. O rapaz era Christopher Morcom, com quem ele estudava e trabalhava junto em projetos científicos. Porém, Morcom morreu repentinamente em 1930, vítima de uma tuberculose bovina. 
O King’s College, de Cambridge, uma das faculdades que compõe a famosa universidade britânica, admitiu Turing quando ele tinha 19 anos. Ele se graduou em matemática com honras no ano de 1934. 
Dois anos depois, ele partiu para um novo desafio acadêmico, dessa vez na Universidade de Princeton, nos Estados Unidos, de onde saiu PhD em 1938. Nesta época, ele desenvolveu alguns dos estudos que, eventualmente, seriam importantes para suas contribuições científicas, como a criptologia, a Máquina de Turing e o multiplicador binário eletromecânico. 
Por seu alto conhecimento no assunto, o governo da Inglaterra chamou Turing para voltar a sua terra natal e trabalhar com a quebra de enigmas e decodificação de mensagens para o governo britânico.
O dia 23 de junho marca o nascimento de uma das mentes humanas mais importantes do progresso tecnológico e científico de todos os tempos: o matemático inglês Alan Mathison Turing (1912 – 1954), considerado o “pai da computação”.
Turing foi uma das primeiras pessoas a pensar nos computadores como um sistema capaz de responder a qualquer tipo de problema, segundo a revista científica Nature em seu artigo “Turing at 100: Legacy of a Universal Mind” (“Turing aos 100 anos: Legado de uma Mente Universal”, em tradução livre).
O matemático fez dessa pergunta uma hipótese interessante para ser investigada no início da década de 1950, sendo um dos primeiros cientistas do mundo a questionar tal possibilidade. 
De acordo com a enciclopédia Britannica (serviço de dados voltado para a educação do Reino Unido), todos os computadores modernos são, em essência, o produto do avanço tecnológico promovido por Turing.
O primeiro trabalho substancial no campo da inteligência artificial ocorreu em meados do século 20, por meio de uma decodificação que levaria o nome de "Máquina de Turing".
Em 1935, informa a Britannica, o pesquisador desenvolveu esse modelo de computação que consistia em uma memória e um scanner que tinham a tarefa de identificar e ler uma série de símbolos espalhados em uma fita que se movia para frente e para trás. 
Assim, a máquina operava e estudava essa série de símbolos a fim de interpretá-los e modificar seu próprio algoritmo de acordo com as instruções dispostas em sua memória. 
Para Turing, a inteligência computacional do futuro deveria ser uma máquina capaz de aprender com a experiência. O caminho para conseguir isso era permitir que uma máquina inteligente alterasse as próprias instruções fornecidas por seu mecanismo, defendia o matemático.
Esse princípio tornou-se um fundamento da teoria moderna da computação. Se hoje os computadores são algo comum na realidade humana, muito disso se deve às contribuições de pesquisadores como Turing, complementa a fonte britânica.
Na Segunda Guerra Mundial (1939-1945), a Alemanha Nazista usava criptografia para se comunicar através de mensagens, impedindo, assim, que seus inimigos decifrassem a localização das tropas alemãs, as estratégias de guerra e o avanço territorial do país, como explica um artigo da National Geographic Espanha. 
Esse computador, chamado de "Enigma", foi desenvolvido por Arthur Scherbius, um engenheiro alemão, e se baseava no envio de mensagens criptografadas que alteravam a forma, mas não o conteúdo, a cada 24 horas. O objetivo era evitar que as criptografias fossem decifradas em caso de interceptação das mensagens pelos inimigos.
No final de 1939, Turing estava trabalhando na sede de comunicações do governo do Reino Unido (a nação fez parte do grupo conhecido como Aliados, que enfrentou a Alemanha Nazista).
Juntamente com seu amigo e matemático Gordon Welchman, ele desenvolveu a contraofensiva tecnológica que permitiu aos Aliados decifrarem o código com o qual os alemães planejavam suas estratégias. 
Esse precursor dos computadores digitais programáveis foi batizado de “Bombe”, nome derivado de uma palavra polonesa que designa um sabor de sorvete. 
De acordo com a National Geographic Espanha, em 1942, mais de 40 mil mensagens criptografadas dos nazistas foram interceptadas, das quais duas eram decifradas a cada minuto.
Turing ajudou a encurtar a guerra na Europa em um período entre dois a quatro anos, salvando assim quatorze milhões de vidas, reconheceu Winston Churchill, primeiro-ministro do Reino Unido durante a maior parte da Segunda Guerra Mundial, e segundo explica um artigo da National Geographic Espanha
"""

# Configuração
API_KEY = 'COLOQUE SUA CHAVE AQUI'


In [ ]:
# ======================================================================
# EXTRAINDO ONTOLOGIA DO TEXTO
# ======================================================================
dados_brutos = extrair_ontologia_com_llm (
        texto_exemplo, 
        API_KEY
)


In [ ]:
# ======================================================================
# CONVERTER ESTRUTURA
# ======================================================================
dados = converter_dados_para_estrutura(dados_brutos)
print("# ======================================================================")
print("# RESULTADO DA CONVERSÃO DA ESTRUTURA")
print("# ======================================================================")
print(f"# Classes identificadas ....: {len(dados['classes'])}")
print(f"# Entidades extraídas ......: {sum(len(v) for v in dados['entidades'].values())}")
print(f"# Relações encontradas .....: {len(dados['relacoes'])}")
print("# ======================================================================")


In [ ]:
# ======================================================================
# CRIANDO ONTOLOGIA RDF
# ======================================================================
grafo, namespace, propriedades = criar_ontologia_rdf(
    dados['classes'],
    dados['entidades'],
    dados['relacoes']
)

In [ ]:
# ======================================================================
# RESUMO DA ONTOLOGIA EXTRAÍDA COM LLM - CLASSES
# ======================================================================
exibir_resumo_classe (
        dados['classes'], 
        dados['entidades'], 
        dados['relacoes']
)


In [ ]:
# ======================================================================
# RESUMO DA ONTOLOGIA EXTRAÍDA COM LLM - ENTIDADES POR CLASSE
# ======================================================================
exibir_resumo_entidade_classe (
        dados['classes'], 
        dados['entidades'], 
        dados['relacoes']
)


In [ ]:
# ======================================================================
# SALVANDO ONTOLOGIA
# ======================================================================
print("# ======================================================================")
print("# SALVANDO A ONTOLOGIA EM ARQUIVO TTL")
print("# ======================================================================")
DIR_DATASET = "C:\\Users\\alexa\\Atividades\\LocalDeepSeek\\Dataset\\"
salvar_ontologia(grafo, DIR_DATASET + 'ontologia_funcional.ttl')
print("# ======================================================================")


In [ ]:
# ====================================================================== 
# FIM DO PROGRAMA
# ====================================================================== 